In [1]:
import torch
import numpy as np
from torch_geometric.datasets import QM9
from tqdm import tqdm 

def extract_focal_neighborhoods(data, max_neighbors=7, d_max=5.0):
    num_atoms = data.pos.shape[0]
    dist_matrix = torch.cdist(data.pos, data.pos)
    molecule_neighborhoods = []
    
    for i in range(num_atoms):
        distances = dist_matrix[i]
        sorted_indices = torch.argsort(distances)
        sorted_distances = distances[sorted_indices]
        
        valid_mask = sorted_distances <= d_max
        valid_indices = sorted_indices[valid_mask]
        
        num_to_take = min(len(valid_indices), max_neighbors + 1)
        neighbor_indices = valid_indices[:num_to_take]
        
        subgraph_z = data.z[neighbor_indices].float()
        subgraph_pos = data.pos[neighbor_indices]
        
        if num_to_take < max_neighbors + 1:
            pad_size = (max_neighbors + 1) - num_to_take
            pad_z = torch.zeros(pad_size, dtype=torch.float32)
            pad_pos = torch.zeros((pad_size, 3), dtype=torch.float32)
            subgraph_z = torch.cat([subgraph_z, pad_z])
            subgraph_pos = torch.cat([subgraph_pos, pad_pos])
        
        molecule_neighborhoods.append({
            'atomic_numbers': subgraph_z, 
            'coordinates': subgraph_pos   
        })
    return molecule_neighborhoods

def encode_diagram_angles(neighborhoods, d_max=5.0):
    processed = []
    for nb in neighborhoods:
        Z_num = nb['atomic_numbers']
        pos = nb['coordinates']
        
        focal_pos = pos[0]
        rel_pos = pos - focal_pos
        x, y, z = rel_pos[:, 0], rel_pos[:, 1], rel_pos[:, 2]
        
        d = torch.sqrt(x**2 + y**2 + z**2)
        theta = torch.acos(torch.clamp(y / (d + 1e-7), -1.0, 1.0))
        phi = torch.atan2(x, z)
        
        is_ghost = (Z_num == 0)
        d = d.masked_fill(is_ghost, 0.0)
        theta = theta.masked_fill(is_ghost, 0.0)
        phi = phi.masked_fill(is_ghost, 0.0)
        
        # CORRECTED: Scaled to 2*pi and using d_max
        Z_scaled = (Z_num / 10.0) * (2 * np.pi)
        d_scaled = (d / d_max) * (2 * np.pi)
        
        theta_scaled = theta  
        phi_scaled = (phi + np.pi) / 2.0  
        phi_scaled = phi_scaled.masked_fill(is_ghost, 0.0)
        
        features = torch.stack([Z_scaled, d_scaled, theta_scaled, phi_scaled], dim=1)
        processed.append(features.flatten()) 
        
    return torch.stack(processed)

# Process Data
print("Loading raw QM9...")
dataset = QM9(root='./data/QM9')
processed_dataset = []
target_property_idx = 4 

num_molecules_to_process = 10

for i in tqdm(range(num_molecules_to_process), desc="Processing Molecules"):
    data = dataset[i]
    target_val = data.y[0, target_property_idx]
    
    neighborhoods = extract_focal_neighborhoods(data, d_max=5.0)
    quantum_features = encode_diagram_angles(neighborhoods, d_max=5.0)
    processed_dataset.append((quantum_features, target_val))

train_size = int(0.8 * len(processed_dataset))
training_data = processed_dataset[:train_size]
test_data = processed_dataset[train_size:]

C:\Users\Sriram Nangunoori\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading raw QM9...


Processing Molecules: 100%|██████████| 10/10 [00:00<00:00, 902.68it/s]


In [1]:
!pip install qiskit-algorithms

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import torch.nn as nn
import torch.optim as optim
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

print("--- Initializing Pure Quantum Circuit (384 Params) ---")
num_qubits, num_atoms, num_layers = 16, 8, 8

x_inputs = ParameterVector('x', 32)
theta_weights = ParameterVector('t', 384) 

pqc = QuantumCircuit(num_qubits)

# CORRECTED: Feature Map strictly uses single-qubit rotations
for i in range(num_atoms):
    qA, qB = 2*i, 2*i + 1
    
    # Qubit A: Encode Atom Type (Z) and Distance (d) 
    pqc.rx(x_inputs[i*4 + 0], qA)
    pqc.ry(x_inputs[i*4 + 1], qA)
    
    # Qubit B: Encode Polar (theta) and Azimuthal (phi) angles
    pqc.rx(x_inputs[i*4 + 2], qB)
    pqc.ry(x_inputs[i*4 + 3], qB)

# Ansatz
weight_idx = 0
for layer in range(num_layers):
    for q in range(num_qubits):
        pqc.rz(theta_weights[weight_idx], q)
        pqc.ry(theta_weights[weight_idx+1], q)
        pqc.rz(theta_weights[weight_idx+2], q)
        weight_idx += 3
    for q in range(num_qubits):
        pqc.cx(q, (q + 1) % num_qubits)

observables = [SparsePauliOp("I" * (15 - i) + "Z" + "I" * i) for i in range(num_qubits)]
estimator = StatevectorEstimator()

# CORRECTED: Define explicit parameter order for the Estimator
all_params = list(x_inputs) + list(theta_weights)

theta = np.random.uniform(-np.pi, np.pi, 384)
quantum_lr = 0.05 

classical_mlp = nn.Sequential(
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

mlp_optimizer = optim.Adam(classical_mlp.parameters(), lr=0.01)
loss_fn = nn.L1Loss()

--- Initializing Pure Quantum Circuit (384 Params) ---


In [4]:
# ==========================================
# 3. THE ISOLATED TRAINING LOOP (CORRECTED)
# ==========================================
print("\n=== Starting Isolated Hybrid Training ===")
epochs = 2

for epoch in range(epochs):
    for mol_idx, (X_molecule, y_target) in enumerate(training_data):
        
        X_np = X_molecule.numpy()
        N_atoms = X_np.shape[0]
        
        print(f"\n--- Epoch {epoch+1} | Molecule {mol_idx+1} ---")
        
        # ---------------------------------------------------------
        # STEP 3: READOUT (Quantum Forward Pass)
        # ---------------------------------------------------------
        Q_out_list = []
        for a in range(N_atoms):
            bound_vals = np.concatenate([X_np[a], theta])
            
            # FIX: Create a dictionary mapping to force correct parameter binding!
            param_dict = {p: v for p, v in zip(all_params, bound_vals)}
            
            # Pass a 3-element tuple: (circuit, observables, parameter_dict)
            job = estimator.run([(pqc, observables, param_dict)])
            Q_out_list.append(job.result()[0].data.evs)
            
        Q_tensor = torch.tensor(np.array(Q_out_list), requires_grad=True, dtype=torch.float32)

        # ---------------------------------------------------------
        # STEP 4: CLASSICAL MLP & LOSS
        # ---------------------------------------------------------
        mol_embedding = torch.sum(Q_tensor, dim=0, keepdim=True)
        prediction = classical_mlp(mol_embedding).squeeze()
        
        y_tensor = y_target.clone().detach().float().squeeze()
        loss = loss_fn(prediction, y_tensor)
        print(f"[DEBUG] Prediction: {prediction.item():.4f} | Loss: {loss.item():.4f}")

        # ---------------------------------------------------------
        # STEP 5: SEPARATED TRAINING & GRADIENTS
        # ---------------------------------------------------------
        mlp_optimizer.zero_grad()
        loss.backward() 
        mlp_optimizer.step() 
        
        dL_dQ = Q_tensor.grad.numpy() 
        print(f"[DEBUG] MLP updated via PyTorch. Starting pure NumPy Parameter Shift...")
        
        # 5b. Pure NumPy Parameter Shift Rule
        manual_theta_grads = np.zeros(384)
        
        for p in range(384):
            theta_plus = theta.copy()
            theta_plus[p] += np.pi / 2.0
            
            theta_minus = theta.copy()
            theta_minus[p] -= np.pi / 2.0
            
            dQ_dtheta_p = np.zeros((N_atoms, 16))
            
            for a in range(N_atoms):
                val_plus = np.concatenate([X_np[a], theta_plus])
                val_minus = np.concatenate([X_np[a], theta_minus])
                
                # FIX: Use dictionary mappings for the shifted evaluations too
                bind_plus = {param: val for param, val in zip(all_params, val_plus)}
                bind_minus = {param: val for param, val in zip(all_params, val_minus)}
                
                job = estimator.run([
                    (pqc, observables, bind_plus),
                    (pqc, observables, bind_minus)
                ])
                res_plus = job.result()[0].data.evs
                res_minus = job.result()[1].data.evs
                
                dQ_dtheta_p[a] = (res_plus - res_minus) / 2.0
            
            manual_theta_grads[p] = np.sum(dL_dQ * dQ_dtheta_p)
            
            if p % 1 == 0:
                print(f"        [DEBUG] Shifted param {p}/384... NumPy Gradient = {manual_theta_grads[p]:.6f}")

        # 5c. Pure NumPy Weight Update
        theta -= quantum_lr * manual_theta_grads
        print(f"[DEBUG] Quantum parameters updated manually via NumPy.")


=== Starting Isolated Hybrid Training ===

--- Epoch 1 | Molecule 1 ---


KeyboardInterrupt: 

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# 1. Create the Quantum Neural Network (QNN)
# This automatically handles the parameter mapping we struggled with earlier!
qnn = EstimatorQNN(
    circuit=pqc,
    observables=observables,
    input_params=x_inputs,
    weight_params=theta_weights
)

# 2. Wrap the QNN into a PyTorch Layer
# We pass random initial weights for the 384 theta parameters
initial_weights = np.random.uniform(-np.pi, np.pi, 384)
quantum_layer = TorchConnector(qnn, initial_weights=initial_weights)

# 3. Build the Hybrid Module
class Quantum3DGraphModel(nn.Module):
    def __init__(self, q_layer):
        super().__init__()
        self.quantum_layer = q_layer
        self.classical_mlp = nn.Sequential(
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        # 1. Quantum Pass: Input is [8, 32]. Output is [8, 16] (16 Pauli-Z expected values per atom)
        quantum_out = self.quantum_layer(x)
        
        # 2. Aggregation: Sum up the atom embeddings to get the molecule embedding [1, 16]
        mol_embedding = torch.sum(quantum_out, dim=0, keepdim=True)
        
        # 3. Classical Pass: Predict the final property
        prediction = self.classical_mlp(mol_embedding)
        return prediction.squeeze()

# Instantiate the full hybrid model
hybrid_model = Quantum3DGraphModel(quantum_layer)

# One optimizer to rule them all (Updates both Quantum and Classical weights!)
optimizer = optim.Adam(hybrid_model.parameters(), lr=0.01)
loss_fn = nn.L1Loss()

print("Hybrid PyTorch-Qiskit Model Successfully Built!")

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Hybrid PyTorch-Qiskit Model Successfully Built!


In [ ]:
import time
import torch

print("\n=== Starting Heavily Tracked Hybrid Training ===")
epochs = 2

for epoch in range(epochs):
    print(f"\n{'='*40}")
    print(f"--- STARTING EPOCH {epoch+1} ---")
    print(f"{'='*40}")
    
    epoch_loss = 0.0
    
    for mol_idx, (X_molecule, y_target) in enumerate(training_data):
        print(f"\n[DEBUG] --- Processing Molecule {mol_idx+1}/{len(training_data)} ---")
        
        # 1. Zero gradients
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | Zeroing gradients...")
        optimizer.zero_grad()
        
        # 2. Forward Pass
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | STARTING Forward Pass (Quantum Simulation + MLP)...")
        start_forward = time.time()
        
        prediction = hybrid_model(X_molecule.float())
        
        end_forward = time.time()
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | FINISHED Forward Pass in {end_forward - start_forward:.2f} seconds!")
        
        # 3. Format target and Loss
        y_tensor = y_target.clone().detach().float().squeeze()
        loss = loss_fn(prediction, y_tensor)
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | Loss Calculated: {loss.item():.4f}")
        
        # 4. Backward Pass (The Heavy Math)
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | STARTING Backward Pass (Calculating Quantum Gradients)...")
        start_backward = time.time()
        
        loss.backward()
        
        end_backward = time.time()
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | FINISHED Backward Pass in {end_backward - start_backward:.2f} seconds!")
        
        # 5. Optimizer Step
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | Updating weights (Optimizer Step)...")
        optimizer.step()
        
        epoch_loss += loss.item()
        print(f"[RESULT] Molecule {mol_idx+1} | Target: {y_tensor.item():.4f} | Pred: {prediction.item():.4f} | Loss: {loss.item():.4f}")
        
    print(f"\n==> Epoch {epoch+1} Average Loss: {epoch_loss / len(training_data):.4f}")


=== Starting Heavily Tracked Hybrid Training ===

--- STARTING EPOCH 1 ---

[DEBUG] --- Processing Molecule 1/8 ---
[DEBUG] 11:39:35 | Zeroing gradients...
[DEBUG] 11:39:35 | STARTING Forward Pass (Quantum Simulation + MLP)...
[DEBUG] 11:39:48 | FINISHED Forward Pass in 12.99 seconds!
[DEBUG] 11:39:48 | Loss Calculated: 13.7860
[DEBUG] 11:39:48 | STARTING Backward Pass (Calculating Quantum Gradients)...


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Use the modern Qiskit 1.0+ imports!
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.gradients import SPSAEstimatorGradient

# 1. Initialize the modern V2 Estimator
estimator = StatevectorEstimator()

# 2. Initialize the SPSA Gradient (The time-saver!)
# Epsilon controls the size of the random parameter shifts. 
spsa_grad = SPSAEstimatorGradient(estimator, epsilon=0.01)

# 3. Create the QNN and explicitly pass the fast SPSA gradient
qnn = EstimatorQNN(
    circuit=pqc,
    observables=observables,
    input_params=x_inputs,
    weight_params=theta_weights,
    estimator=estimator,
    gradient=spsa_grad
)

# 4. Wrap the QNN into a PyTorch Layer
initial_weights = np.random.uniform(-np.pi, np.pi, 384)
quantum_layer = TorchConnector(qnn, initial_weights=initial_weights)

print("Fast SPSA Hybrid QNN Successfully Built!")

Fast SPSA Hybrid QNN Successfully Built!


In [5]:
import time
import torch

print("\n=== Starting Heavily Tracked Hybrid Training ===")
epochs = 2

for epoch in range(epochs):
    print(f"\n{'='*40}")
    print(f"--- STARTING EPOCH {epoch+1} ---")
    print(f"{'='*40}")
    
    epoch_loss = 0.0
    
    for mol_idx, (X_molecule, y_target) in enumerate(training_data):
        print(f"\n[DEBUG] --- Processing Molecule {mol_idx+1}/{len(training_data)} ---")
        
        # 1. Zero gradients
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | Zeroing gradients...")
        optimizer.zero_grad()
        
        # 2. Forward Pass
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | STARTING Forward Pass (Quantum Simulation + MLP)...")
        start_forward = time.time()
        
        prediction = hybrid_model(X_molecule.float())
        
        end_forward = time.time()
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | FINISHED Forward Pass in {end_forward - start_forward:.2f} seconds!")
        
        # 3. Format target and Loss
        y_tensor = y_target.clone().detach().float().squeeze()
        loss = loss_fn(prediction, y_tensor)
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | Loss Calculated: {loss.item():.4f}")
        
        # 4. Backward Pass (The Heavy Math)
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | STARTING Backward Pass (Calculating Quantum Gradients)...")
        start_backward = time.time()
        
        loss.backward()
        
        end_backward = time.time()
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | FINISHED Backward Pass in {end_backward - start_backward:.2f} seconds!")
        
        # 5. Optimizer Step
        print(f"[DEBUG] {time.strftime('%H:%M:%S')} | Updating weights (Optimizer Step)...")
        optimizer.step()
        
        epoch_loss += loss.item()
        print(f"[RESULT] Molecule {mol_idx+1} | Target: {y_tensor.item():.4f} | Pred: {prediction.item():.4f} | Loss: {loss.item():.4f}")
        
    print(f"\n==> Epoch {epoch+1} Average Loss: {epoch_loss / len(training_data):.4f}")


=== Starting Heavily Tracked Hybrid Training ===

--- STARTING EPOCH 1 ---

[DEBUG] --- Processing Molecule 1/8 ---
[DEBUG] 12:15:37 | Zeroing gradients...


NameError: name 'optimizer' is not defined